<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/HTSModelDataPrep26082026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# This script begins with tables exported from the National Data warehouse and
#concludes with the creation of ML-ready datasets that are  then read in by
# model training scripts. By ML-ready datasets, we mean datasets in which each
# row is an observation with a labeled outcome.

# The sript proceeds through three steps:
# 1) Combining and filtering original tables
# 2) Data Cleaning and feature generation
# 3) Missing data imputation

# First, load necessary packages
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from statsmodels.imputation.mice import MICEData


                       **  Combine and filter original tables**

In [3]:
# This script assumes user is accessing tables extracted and loaded from ODS Database.

#Load tables
eligibility = pd.read_csv('eligibility.csv', low_memory=False)
tests = pd.read_csv('tests.csv', low_memory=False)
xwalk = pd.read_csv('ActiveEMRSites_07152026.csv', low_memory=False)
clients = pd.read_csv('clients.csv', low_memory=False)

In [4]:
# Preview content of tables
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")

rows,cols = xwalk.shape
print(f"There are {rows} rows and {cols} columns in the xwalk table")

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

There are 2709027 rows and 87 columns in the eligibility table
There are 3157391 rows and 34 columns in the tests table
There are 2456 rows and 19 columns in the xwalk table
There are 12199267 rows and 6 columns in the clients table


In [ ]:
eligibility = pd.read_csv('eligibility.csv', low_memory=False)

In [ ]:
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

There are 2709027 rows and 87 columns in the eligibility table


In [6]:
# Tests
# Filter tests to results that are positive or negative (exclude inconclusive)
display(tests.columns)

tests['FinalTestResult']=tests['FinalTestResult'].str.upper()

tests = tests[
    (tests['FinalTestResult'] == 'POSITIVE') |
    (tests['FinalTestResult'] == 'NEGATIVE')
]

# Select columns to keep and remove duplicates(We are keeping all columns)
cols_to_keep = ['SiteCode','PatientPk','EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'ClientTestedAs','CoupleDiscordant','PriorityPopulationType']
tests = tests[cols_to_keep]
tests = tests.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")


Index(['FacilityName', 'SiteCode', 'PatientPk', 'Emr', 'Project',
       'EncounterId', 'TestDate', 'EverTestedForHiv', 'MonthsSinceLastTest',
       'ClientTestedAs', 'EntryPoint', 'TestStrategy', 'TestResult1',
       'TestResult2', 'TestResult3', 'FinalTestResult', 'PatientGivenResult',
       'TbScreening', 'ClientSelfTested', 'CoupleDiscordant', 'TestType',
       'Consent', 'Setting', 'Approach', 'HtsRiskCategory', 'HtsRiskScore',
       'PatientPKHash', 'OtherReferredServices', 'ReferredForServices',
       'ReferredServices', 'LoadDate', 'RecordUUID', 'PriorityPopulationType',
       'DateExtracted'],
      dtype='object')

There are 2713546 rows and 8 columns in the tests table


In [8]:
tests['FinalTestResult'].value_counts(dropna=False)

,count
FinalTestResult,
NEGATIVE,2643390
POSITIVE,70156


In [9]:
#Clients
# Select columns to keep and remove duplicates. We are keeping all columns in clients dataset
display(clients.columns)

clients = clients.drop_duplicates(subset= ["SiteCode","PatientPk"])

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

Index(['PatientPk', 'SiteCode', 'Dob', 'Sex', 'MaritalStatus',
       'PatientDisabled'],
      dtype='object')

There are 12199267 rows and 6 columns in the clients table


In [10]:
# Eligibility
# Convert visitdate from character to date and filter to between April 2025 and June 2026
print(eligibility["VisitDate"].dtype)
eligibility['VisitDate'] = pd.to_datetime(eligibility['VisitDate'].astype(str).str[:10])

eligibility = eligibility[eligibility['VisitDate'] >= '2025-04-01']
eligibility = eligibility[eligibility['VisitDate'] <= '2026-06-30']

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

object
There are 2509280 rows and 87 columns in the eligibility table


In [11]:
# Select columns to keep and remove duplicates(Calculation of missingness and variance done separately)
cols = eligibility.columns
#print(cols)
cols_to_keep = ['SiteCode', 'PatientPk','VisitDate', 'PopulationType', 'KeyPopulation', 'PriorityPopulation',
                'IsHealthWorker','RelationshipWithContact', 'TestedHIVBefore','ResultOfHIV','EverHadSex',
                'SexuallyActive', 'NewPartner', 'PartnerHIVStatus','MultiplePartners', 'NumberOfPartners',
                'AlcoholSex', 'MoneySex','CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant','BreastfeedingMother',
                'ExperiencedViolenceScreening','CurrentlyOnPrep','TraditionalProcedures','MothersStatus',
                'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices','ViolenceScreeningType','Disability', 'DisabilityType']
eligibility = eligibility[cols_to_keep]
eligibility = eligibility.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

There are 2179395 rows and 34 columns in the eligibility table


In [12]:
eligibility.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType'],
      dtype='object')

In [13]:
xwalk.columns

Index(['MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR', 'EMR_Status', 'HTS_Use',
       'HTS_Deployment', 'Project', 'LoadDate', 'InfrastructureType',
       'KEPH_Level', 'KMPDC_reg_no', 'Ward'],
      dtype='object')

In [14]:
# Select columns to keep and remove duplicates
cols = xwalk.columns

cols_to_keep =['MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR','Project', 'KEPH_Level','Ward']
xwalk = xwalk[cols_to_keep]
xwalk = xwalk.drop_duplicates(subset=["MFL_Code"])

rows,cols = xwalk.shape
print(f"There are {rows} rows and {cols} columns in the xwalk table")

There are 2456 rows and 13 columns in the xwalk table


In [15]:
# Join the three tables on SiteCode and PatientPK
merged = pd.merge(eligibility, clients, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, tests, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, xwalk, left_on=["SiteCode"], right_on=["MFL_Code"], how="left")

rows,cols = merged.shape
print(f"There are {rows} rows and {cols} columns in the merged table")

#download the merged file
import requests
import csv
merged.to_csv('merged.csv', index=False)

There are 2179395 rows and 57 columns in the merged table


In [16]:
#Rename the merged dataset to hts
hts = merged.copy()


rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2179395 rows and 57 columns in the hts table


In [17]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'Dob', 'Sex',
       'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agen

Quick Checks

In [18]:
#Filter the data to exclude ResultofHIV positive
hts['ResultOfHIV'] = hts['ResultOfHIV'].str.upper()
hts = hts[hts['ResultOfHIV'] != 'POSITIVE']

In [19]:
rows,cols = hts.shape
print(f" There are {rows} Rows and {cols} Columns in HTS dataset")

 There are 2177068 Rows and 57 Columns in HTS dataset


In [20]:
#Keep variable...PatientDisabled. Drop Disability & DisabilityType because of high missingness
pd.crosstab(
    clients['PatientDisabled'],
    tests['FinalTestResult'],
    dropna=False
)

FinalTestResult,NEGATIVE,POSITIVE
PatientDisabled,,
No,1705900,46466
Yes,933897,23579
NaN,3593,111


In [21]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'Dob', 'Sex',
       'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agen

In [22]:
# Identify missing or no-variance variables to exclude
# Get breakdown by variable
# Calculate missingness for all columns and display only those with >50% missing

missing_percent = hts.isnull().mean() * 100

high_missing = missing_percent[missing_percent > 50].sort_values(ascending=False)

print(high_missing)

Project                    100.000000
DisabilityType              99.782000
ResultOfHIVSelf             98.705599
Disability                  98.142042
ViolenceScreeningType       97.542107
MothersStatus               97.403572
PriorityPopulation          97.032063
PriorityPopulationType      96.123502
CoupleDiscordant            94.762405
KeyPopulation               93.028100
RelationshipWithContact     90.072382
NumberOfPartners            88.957120
MonthsSinceLastTest         61.908723
dtype: float64


In [ ]:
print(type(hts))

<class 'pandas.core.frame.DataFrame'>


In [23]:
variables_to_drop = ["Project", "SDP_Agency",'Owner','Disability','DisabilityType','RelationshipWithContact' ]

hts= hts.drop(columns=variables_to_drop, errors="ignore")

#rows,cols = hts.shape
#print(f"There are {rows} rows and {cols} columns in the hts table")

In [24]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Latitude', 'Longitude', 'SDP',
       'EMR', 'KEPH_Level', 'Ward'],
      dtype='object')

In [26]:
#download the hts file
hts.to_csv('hts.csv', index=False)

from google.colab import files
files.download('hts.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                                 **   Data cleaning and feature generation **

In [50]:
hts = pd.read_csv('hts.csv',low_memory=False)
#display(hts.head())
rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2177068 rows and 51 columns in the hts table


In [51]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Latitude', 'Longitude', 'SDP',
       'EMR', 'KEPH_Level', 'Ward'],
      dtype='object')

# Demographic Features

Age

In [52]:
# Create adult/child flag for checking patterns more easily
# Create VDate as date field, DOB as date field, and Age

hts['VDate'] = pd.to_datetime(hts['VisitDate'].astype(str).str[:10])
hts['DOB'] = pd.to_datetime(hts['Dob'].astype(str).str[:10], errors='coerce')

# Calculate Age
hts['Age'] = (hts['VDate'] - hts['DOB']).dt.days // 365

# Filter out erroneous ages (Age > 0 and Age < 100)
hts = hts[hts['Age'] > 0]
hts = hts[hts['Age'] < 100]

# Create cohort column
hts['cohort'] = np.where(hts['Age'] >= 15, "Adult", "Child")

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after age filtering")

There are 2176785 rows and 55 columns in the hts table after age filtering


Sex

In [53]:
#gender
#display(hts['Sex'].value_counts(dropna=False))

hts['Sex'] = hts['Sex'].astype(str).str.upper()
hts['Sex'] = hts['Sex'].str.replace('FEMALE', 'F')
hts['Sex'] = hts['Sex'].str.replace('MALE', 'M')

#display(hts['Sex'].value_counts(dropna=False))


In [54]:
hts['FinalTestResult'].value_counts(dropna=False)
hts = hts.dropna(subset=['FinalTestResult'])
hts['FinalTestResult'].value_counts(dropna=False)

,count
FinalTestResult,
NEGATIVE,2065547
POSITIVE,53521


In [55]:
#check positivity rate
tab = pd.crosstab(
    hts['Sex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ positivity rate is higher in males

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
Sex,,,
F,1426057,33915,2.32299
M,639490,19606,2.97468


Marital Status

In [56]:
hts['MaritalStatus'].value_counts(dropna=False)

,count
MaritalStatus,
MARRIED MONOGAMOUS,1010193
Single,530080
NaN,423817
MARRIED POLYGAMOUS,45592
DIVORCED,38392
SINGLE,24318
WIDOWED,23165
COHABITING,15763
UNKNOWN,4978


In [ ]:
#Inspect other record
#hts.loc[hts['MaritalStatus']=='OTHER'].to_dict('records')

In [57]:
#Replace Other with Unknown
hts['MaritalStatus'] = hts['MaritalStatus'].str.upper()
hts['MaritalStatus']= hts['MaritalStatus'].replace('OTHER', 'UNKNOWN')

pd.crosstab(
hts['MaritalStatus'],
hts['EMR'],
dropna=False
)

EMR,AMRS,ECare,KenyaEMR,NaN
MaritalStatus,,,,
COHABITING,0,0,14660,1103
DIVORCED,0,48,36239,2105
MARRIED MONOGAMOUS,0,15686,959348,35159
MARRIED POLYGAMOUS,0,213,44344,1035
NO,0,0,158,0
SEPARATED,0,1496,101,240
SINGLE,0,12171,503298,38929
UNKNOWN,0,0,4746,233
WIDOWED,0,187,22252,726


In [58]:
#Group categories in Marital Status
ms = hts['MaritalStatus'].str.upper()
hts['MaritalStatus'] = np.select(
    [
        (hts['Age'] < 15) & hts['MaritalStatus'].notna(),
        ms.eq('UNKNOWN'),
        ms.isin(['DIVORCED', 'SEPARATED']),
        ms.eq('MARRIED POLYGAMOUS'),
        ms.isin(['SINGLE', 'NO']),
        ms.isin(['MARRIED MONOGAMOUS', 'COHABITING', 'YES']),
        ms.eq('WIDOWED')
    ],
    [
        'MINOR',
        'UNKNOWN',
        'DIVORCED',
        'POLYGAMOUS',
        'SINGLE',
        'MARRIED',
        'WIDOWED'
    ],
    default=None
)

hts['MaritalStatus'] = hts['MaritalStatus'].replace({None: np.nan})

In [59]:
hts['MaritalStatus'].value_counts(dropna=False)

,count
MaritalStatus,
MARRIED,1023674
SINGLE,492428
NaN,423817
MINOR,65504
POLYGAMOUS,45415
DIVORCED,40177
WIDOWED,23137
UNKNOWN,4916


In [60]:
#Check positivity rate
tab = pd.crosstab(
    hts['MaritalStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# Positivity rate is higher in Divorced and widowed at 13%

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
MaritalStatus,,,
DIVORCED,34970,5207,12.960151
MARRIED,1001461,22213,2.169929
MINOR,64297,1207,1.842636
POLYGAMOUS,42893,2522,5.553231
SINGLE,480535,11893,2.415175
UNKNOWN,4902,14,0.284784
WIDOWED,20140,2997,12.953278
NaN,416349,7468,1.762081


In [61]:
#Check distribution of the missing values
tab = pd.crosstab(
    index=[hts['EMR'], hts['MaritalStatus']],
    columns=hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)

FinalTestResult         NEGATIVE  POSITIVE  PositivityRate
EMR      MaritalStatus                                    
AMRS     DIVORCED              0         0             NaN
         MARRIED               0         0             NaN
         MINOR                 0         0             NaN
         POLYGAMOUS            0         0             NaN
         SINGLE                0         0             NaN
         UNKNOWN               0         0             NaN
         WIDOWED               0         0             NaN
         NaN               38021      1110        2.836626
ECare    DIVORCED           1402       141        9.138043
         MARRIED           15324       342        2.183072
         MINOR              1606        17        1.047443
         POLYGAMOUS          194        19        8.920188
         SINGLE            10261       308        2.914183
         UNKNOWN               0         0             NaN
         WIDOWED             158        29       15.508021
         NaN                   0         0             NaN
KenyaEMR DIVORCED          31419      4872       13.424816
         MARRIED          950538     21316        2.193334
         MINOR             59813      1161        1.904090
         POLYGAMOUS        41716      2454        5.555807
         SINGLE           434664     11059        2.481137
         UNKNOWN            4669        14        0.298954
         WIDOWED           19331      2894       13.021372
         NaN              354667      6147        1.703648
NaN      DIVORCED           2149       194        8.279983
         MARRIED           35599       555        1.535100
         MINOR              2878        29        0.997592
         POLYGAMOUS          983        49        4.748062
         SINGLE            35610       526        1.455612
         UNKNOWN             233         0        0.000000
         WIDOWED             651        74       10.206897
         NaN               23661       211        0.883881

In [62]:
pd.crosstab(
    hts['MaritalStatus'],
    hts['EverHadSex'],
    dropna=False
)


EverHadSex,No,Yes,NaN
MaritalStatus,,,
DIVORCED,0,36235,3942
MARRIED,0,941002,82672
MINOR,0,2562,62942
POLYGAMOUS,0,41171,4244
SINGLE,0,425097,67331
UNKNOWN,0,4595,321
WIDOWED,0,20648,2489
NaN,2864,351443,69510


# Clinical Risk Features

Pregnancy

In [63]:
hts['Pregnant'].value_counts(dropna=False)

,count
Pregnant,
NO,1056822
NaN,722116
YES,316560
No,11125
Declined to answer,10041
Yes,2404


In [64]:
# Pregnant
hts['Pregnant'] = hts['Pregnant'].str.upper()

conditions = [
    hts['Pregnant'].eq('DECLINED TO ANSWER'),
    (hts['Age'] < 10) & hts['Pregnant'].notna(),
    (hts['Sex'].str.upper() == 'M') & hts['Pregnant'].notna(),
    (hts['Age'] > 50) & hts['Pregnant'].notna(),
    hts['Pregnant'].eq('YES'),
    hts['Pregnant'].eq('NO')

]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'NR',
    'YES',
    'NO'

]

hts['Pregnant'] = np.select(conditions, choices, default='None')
hts['Pregnant'] = hts['Pregnant'].replace({'None': np.nan})

#display(hts['Pregnant'].value_counts(dropna=False))


In [65]:
display(hts['Pregnant'].value_counts(dropna=False))

,count
Pregnant,
NO,1004627
NaN,722116
YES,317154
NR,65130
DECLINED,10041


In [66]:
#Check why we have missing values in pregnant
pd.crosstab(
hts['Pregnant'],
hts['EMR'],
dropna=False
)

EMR,AMRS,ECare,KenyaEMR,NaN
Pregnant,,,,
DECLINED,31,0,9566,444
NO,10475,49,937601,56502
NR,531,49,61948,2602
YES,2388,8,301506,13252
NaN,25706,29695,636113,30602


In [67]:
#Check for positivity
tab = pd.crosstab(
    hts['Pregnant'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ low positivity rate amongst pregnant women

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
Pregnant,,,
DECLINED,9833,208,2.071507
NO,981962,22665,2.256061
NR,61651,3479,5.341624
YES,311659,5495,1.732597
NaN,700442,21674,3.001457


BreastfeedingMother

In [68]:
hts['BreastfeedingMother'].value_counts(dropna=False )

,count
BreastfeedingMother,
NO,975807
NaN,755352
YES,369034
No,8104
Declined to answer,6746
Yes,4025


In [71]:
#BreastfeedingMother
hts['BreastfeedingMother'] = hts['BreastfeedingMother'].str.upper()
conditions = [
    hts['BreastfeedingMother'].eq('DECLINED TO ANSWER'),
   (hts['Age'] < 10) & hts['BreastfeedingMother'].notna(),
    (hts['Sex'].str.upper() == 'M') & hts['BreastfeedingMother'].notna(),
    (hts['Age'] > 50) & hts['BreastfeedingMother'].notna(),
    hts['BreastfeedingMother'].eq('YES'),
    hts['BreastfeedingMother'].eq('NO')

]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'NR',
    'YES',
    'NO'

]
hts['BreastfeedingMother'] = np.select(conditions, choices, default='None')
hts['BreastfeedingMother'] = hts['BreastfeedingMother'].replace({'None': np.nan})

In [72]:
hts['BreastfeedingMother'].value_counts(dropna=False)

,count
BreastfeedingMother,
NO,921613
NaN,755352
YES,371805
NR,63552
DECLINED,6746


In [74]:
pd.crosstab(
    hts['BreastfeedingMother'],
    hts['EMR'],
    dropna=False
)

EMR,AMRS,ECare,KenyaEMR,NaN
BreastfeedingMother,,,,
DECLINED,22,0,6350,374
NO,7688,0,861345,52580
NR,423,0,60578,2551
YES,4018,0,352718,15069
NaN,26980,29801,665743,32828


In [75]:
#check positivity rate
tab = pd.crosstab(
    hts['BreastfeedingMother'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ low positivity rate amongst breastfeeding women

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
BreastfeedingMother,,,
DECLINED,6614,132,1.956715
NO,895889,25724,2.791193
NR,60160,3392,5.337362
YES,369854,1951,0.524737
NaN,733030,22322,2.955179


# **HIV Testing History Features**

 EverTestedForHiv

In [76]:
hts['TestedHIVBefore'].value_counts(dropna=False)

,count
TestedHIVBefore,
Yes,1303181
No,807063
NaN,8824


In [77]:
# Ever Tested for HIV
#Standardize the values

hts['TestedHIVBefore'] = hts['TestedHIVBefore'].str.upper()


In [78]:
#Check for positivity rate
tab = pd.crosstab(
    hts['TestedHIVBefore'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # Positivity rate is higher with those ever tested for HIV

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
TestedHIVBefore,,,
NO,781558,25505,3.160224
YES,1276413,26768,2.054051
NaN,7576,1248,14.143246


MonthsSinceLastTest

In [79]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Latitude', 'Longitude', 'SDP',
       'EMR', 'KEPH_Level', 'Ward', 'VDate', 'DOB', 'Age', 'cohort',
       'BreastfeedingM

In [80]:
pd.crosstab(
    hts['EverTestedForHiv'],
    hts['MonthsSinceLastTest'],
    dropna=False
)

MonthsSinceLastTest,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,242.0,243.0,244.0,245.0,248.0,249.0,250.0,260.0,1300.0,NaN
EverTestedForHiv,,,,,,,,,,,,,,,,,,,,,
No,26684,119,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,958069
Yes,18521,47800,53054,101835,43035,37934,65731,18933,23030,18086,...,1,1,3,2,2,2,43,2,1,255709
NaN,3300,42,0,1,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,76096


In [82]:
print((hts['EverTestedForHiv'] != 'YES').sum())

print(((hts['MonthsSinceLastTest'] >= 0) &
       (hts['MonthsSinceLastTest'] <= 6)).sum())

print(((hts['MonthsSinceLastTest'] >= 7) &
       (hts['MonthsSinceLastTest'] <= 12)).sum())

print(((hts['MonthsSinceLastTest'] >= 13) &
       (hts['MonthsSinceLastTest'] <= 24)).sum())

print((hts['MonthsSinceLastTest'] > 24).sum())

2119068
398059
196105
152524
82506


In [83]:
# Months since last test
hts['MonthsSinceLastTest'] = pd.to_numeric(hts['MonthsSinceLastTest'], errors='coerce')
hts['EverTestedForHiv']=hts['EverTestedForHiv'].str.upper()

conditions = [
    hts['EverTestedForHiv'] != 'YES' ,
    (hts['MonthsSinceLastTest'] >= 0) & (hts['MonthsSinceLastTest'] <= 6),
    (hts['MonthsSinceLastTest'] >= 7) & (hts['MonthsSinceLastTest'] <= 12),
    (hts['MonthsSinceLastTest'] >= 13) & (hts['MonthsSinceLastTest'] <= 24),
    hts['MonthsSinceLastTest'] > 24
]

choices = [
    'NR',
    'LASTSIXMONTHS',
    'SEVENTOTWELVE',
    'ONETOTWOYEARS',
    'MORETHANTWOYEARS'
]

hts['MonthsSinceLastTest'] = np.select(conditions, choices, default='None')
hts['MonthsSinceLastTest'] = hts['MonthsSinceLastTest'].replace({'None': np.nan})


#display(hts['MonthsSinceLastTest'].value_counts(dropna=False))

In [84]:
#Check positivity
tab = pd.crosstab(
    hts['MonthsSinceLastTest'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# higher positivity in those tested more than 2years ago at 6%

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
MonthsSinceLastTest,,,
LASTSIXMONTHS,362955,4955,1.346797
MORETHANTWOYEARS,77694,4812,5.832303
NR,1035089,29229,2.746266
ONETOTWOYEARS,147729,4794,3.143133
SEVENTOTWELVE,191398,4704,2.398752
NaN,250682,5027,1.965907


#Behaviour Risk Features

New Partner

In [85]:
hts['NewPartner'].value_counts(dropna=False)

,count
NewPartner,
NO,1205646
YES,442900
NaN,417289
No,24098
Declined to answer,18915
Yes,10220


In [86]:
# New Partner

hts['NewPartner'] = hts['NewPartner'].str.upper()
hts['EverHadSex'] = hts['EverHadSex'].str.upper()

conditions = [
    hts['NewPartner'].eq('DECLINED TO ANSWER'),
    hts['EverHadSex'].eq('NO'),
    (hts['Age'] <15) & hts['NewPartner'].notna(),
    hts['NewPartner'].eq('YES'),
    hts['NewPartner'].eq('NO'),

]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'YES',
    'NO'

]

hts['NewPartner'] = np.select(conditions, choices, default='None')
hts['NewPartner'] = hts['NewPartner'].replace({'None': np.nan})
#display(hts['NewPartner'].value_counts(dropna=False))

In [87]:
display(hts['NewPartner'].value_counts(dropna=False))

,count
NewPartner,
NO,1227484
YES,452220
NaN,414425
DECLINED,18915
NR,6024


In [88]:
#Check for positivity rate
tab = pd.crosstab(
    hts['NewPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 4% positivity for those with new partners

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
NewPartner,,,
DECLINED,18278,637,3.367698
NO,1200263,27221,2.217626
NR,5969,55,0.913015
YES,435270,16950,3.748176
NaN,405767,8658,2.089160


In [89]:
#Check frequency of SexuallyActive vs New partner
hts['SexuallyActive'] = hts['SexuallyActive'].astype(str).str.upper()
cross_tab_sex_partner = pd.crosstab(hts['SexuallyActive'], hts['NewPartner'])
display(cross_tab_sex_partner) # Data to be reviewed are those not sexually active but have new partner

NewPartner,DECLINED,NO,NR,YES
SexuallyActive,,,,
DECLINED TO ANSWER,1033,1664,10,365
NAN,119,5319,18,4635
NO,1690,140777,3285,10902
YES,16073,1079724,2711,436318


Multiple Partners

In [90]:
hts['MultiplePartners'].value_counts(dropna=False)

,count
MultiplePartners,
NO,1343343
NaN,441215
YES,300064
No,24708
Yes,9738


In [92]:
#Standardize Multiple Partner Options
hts['MultiplePartners']= hts['MultiplePartners'].str.upper()

conditions = [
    hts['EverHadSex'].eq('NO'),
    hts['Age'] < 15 & hts['MultiplePartners'].notna(),
    hts['MultiplePartners'].eq('YES'),
    hts['MultiplePartners'].eq('NO')
]

choices = [
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['MultiplePartners'] = np.select(conditions, choices, default='None')
hts['MultiplePartners'] = hts['MultiplePartners'].replace({'None': np.nan})
#display(hts['MultiplePartners'].value_counts(dropna=False))

In [ ]:
display(hts['MultiplePartners'].value_counts(dropna=False))

,count
MultiplePartners,
NO,1368051
NaN,441215
YES,309802


In [93]:
#Check for positivity rate
tab = pd.crosstab(
    hts['MultiplePartners'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 4% positivity for those with multiple partners

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
MultiplePartners,,,
NO,1337825,30226,2.209421
NR,2823,41,1.431564
YES,295914,13888,4.482863
NaN,428985,9366,2.136644


In [94]:
#Quick check Multiplepartner is Yes and Sexually active is No
hts['SexuallyActive'] = hts['SexuallyActive'].str.upper()
nrow_multiple_partners_not_sexually_active = hts[(hts['MultiplePartners'] == 'YES') & (hts['SexuallyActive'] == 'NO')].shape[0]
print(f"Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': {nrow_multiple_partners_not_sexually_active}")

Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': 8229


In [95]:

pd.crosstab(
    index=[hts['cohort'], hts['MultiplePartners']],
    columns=hts['SexuallyActive'],
    margins=True,
    dropna=False
)


SexuallyActive           DECLINED TO ANSWER     NAN      NO      YES  \
cohort MultiplePartners                                                
Adult  NO                              2688    7025  142693  1212909   
       NR                                 0       0    2863        0   
       YES                              312    2238    8211   298624   
       NaN                              174  278488    6953    66662   
Child  NO                                12      15     400     2309   
       NR                                 0       0       1        0   
       YES                                2       4      18      393   
       NaN                              461   27835   51295     6483   
All                                    3649  315605  212434  1587380   

SexuallyActive                 All  
cohort MultiplePartners             
Adult  NO                1365315.0  
       NR                   2863.0  
       YES                309385.0  
       NaN                     NaN  
Child  NO                   2736.0  
       NR                      1.0  
       YES                   417.0  
       NaN                     NaN  
All                      2119068.0

MoneySex

In [96]:
hts['MoneySex'].value_counts(dropna=False)

,count
MoneySex,
NO,1394294
NaN,443736
YES,222839
No,31813
Declined to answer,23863
Yes,2523


In [97]:
#Money Sex
hts['MoneySex'] = hts['MoneySex'].str.upper()

conditions = [
    hts['MoneySex'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <15 & hts['MoneySex'].notna()),
    hts['EverHadSex'].eq('NO'),
    hts['MoneySex'].eq('YES'),
    hts['MoneySex'].eq('NO')
]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['MoneySex'] = np.select(conditions, choices, default='None')
hts['MoneySex'] = hts['MoneySex'].replace({'None': np.nan})

#display(hts['MoneySex'].value_counts(dropna=False))



In [98]:
display(hts['MoneySex'].value_counts(dropna=False))

,count
MoneySex,
NO,1426107
NaN,440872
YES,225362
DECLINED,23863
NR,2864


In [99]:
#Check for positivity rate
tab = pd.crosstab(
    hts['MoneySex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~3.2% positivity rate for those who  have sex for money

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
MoneySex,,,
DECLINED,22989,874,3.662574
NO,1390132,35975,2.522602
NR,2823,41,1.431564
YES,218226,7136,3.166461
NaN,431377,9495,2.153686


In [100]:
# Calculate the number of rows where MoneySex is 'YES' and EverHadSex is 'NO'
num_money_sex_no_sex = hts[
    (hts['MoneySex'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': {num_money_sex_no_sex}")

Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': 0


AlcoholSex

In [101]:
display(hts['AlcoholSex'].value_counts(dropna=False))

,count
AlcoholSex,
Not at all,1417283
NaN,448380
Sometimes,209953
Always,43256
Declined to answer,196


In [102]:
# Alcohol Sex
hts['AlcoholSex'] = hts['AlcoholSex'].str.upper()

conditions = [
    hts['AlcoholSex'].eq('DECLINED TO ANSWER'),
    hts['Age'] <15 & hts['AlcoholSex'].notna(),
    hts['EverHadSex'].eq('NO'),
    hts['AlcoholSex'].eq('ALWAYS'),
    hts['AlcoholSex'].eq('SOMETIMES'),
    hts['AlcoholSex'].eq('NOT AT ALL')
]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'ALWAYS',
    'SOMETIMES',
    'NEVER'
]

hts['AlcoholSex'] = np.select(conditions, choices, default='None')
hts['AlcoholSex'] = hts['AlcoholSex'].replace({'None': np.nan})

#display(hts['AlcoholSex'].value_counts(dropna=False))

In [103]:
display(hts['AlcoholSex'].value_counts(dropna=False))

,count
AlcoholSex,
NEVER,1417283
NaN,445516
SOMETIMES,209953
ALWAYS,43256
NR,2864
DECLINED,196


In [104]:
#check positivity rate
tab = pd.crosstab(
    hts['AlcoholSex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~3.7% positivity rate for those who always have alcohol sex

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
AlcoholSex,,,
ALWAYS,41627,1629,3.765952
DECLINED,192,4,2.040816
NEVER,1382916,34367,2.424851
NR,2823,41,1.431564
SOMETIMES,202350,7603,3.621287
NaN,435639,9877,2.216980


CondomBurst

In [105]:
display(hts['CondomBurst'].value_counts(dropna=False))

,count
CondomBurst,
NO,1412470
NaN,446720
YES,202082
No,29579
Declined to answer,23599
Yes,4618


In [106]:
#Condom Burst
hts['CondomBurst'] = hts['CondomBurst'].str.upper()

conditions = [
    hts['CondomBurst'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <= 9 & hts['CondomBurst'].notna()),
    hts['EverHadSex'].eq('NO'),
    hts['CondomBurst'].eq('YES'),
    hts['CondomBurst'].eq('NO')
]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['CondomBurst'] = np.select(conditions, choices, default='None')
hts['CondomBurst'] = hts['CondomBurst'].replace({'None': np.nan})
#display(hts['CondomBurst'].value_counts(dropna=False))

In [107]:
display(hts['CondomBurst'].value_counts(dropna=False))

,count
CondomBurst,
NO,1442034
NaN,443856
YES,206699
DECLINED,23599
NR,2880


In [108]:
#check positivity rate
tab = pd.crosstab(
    hts['CondomBurst'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3.3.% positivity rate for those who said they have had condom burst

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
CondomBurst,,,
DECLINED,22810,789,3.343362
NO,1405841,36193,2.509858
NR,2838,42,1.458333
YES,199856,6843,3.310611
NaN,434202,9654,2.175030


In [109]:
# Calculate the number of rows where CondomBurst is 'YES' and EverHadSex is 'NO'
num_condom_burst_no_sex = hts[
    (hts['CondomBurst'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': {num_condom_burst_no_sex}")

Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': 0


# Partner Risk Features

PartnerHIVStatus

In [110]:
display(hts['PartnerHIVStatus'].value_counts(dropna=False))


,count
PartnerHIVStatus,
Unknown,1038242
Negative,594936
NaN,416593
Positive,49527
Declined to answer,14901
HIV Negative,3688
HIV Positive,1181


In [111]:
#Replace
hts['PartnerHIVStatus'] = hts['PartnerHIVStatus'].str.upper()
hts['PartnerHIVStatus']=hts['PartnerHIVStatus'].replace('NEGATIVE', 'HIV NEGATIVE')
hts['PartnerHIVStatus']=hts['PartnerHIVStatus'].replace('POSITIVE', 'HIV POSITIVE')

In [112]:
display(hts['PartnerHIVStatus'].value_counts(dropna=False))

,count
PartnerHIVStatus,
UNKNOWN,1038242
HIV NEGATIVE,598624
NaN,416593
HIV POSITIVE,50708
DECLINED TO ANSWER,14901


In [113]:
# PartnerHIVStatus
hts['PartnerHIVStatus'] = hts['PartnerHIVStatus'].str.upper()

conditions = [
    hts['PartnerHIVStatus'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <15) & hts['PartnerHIVStatus'].notna(),
    hts['EverHadSex'].eq('NO'),
    hts['PartnerHIVStatus'].eq('HIV POSITIVE'),
    hts['PartnerHIVStatus'].eq('HIV NEGATIVE'),
    hts['PartnerHIVStatus'].eq('UNKNOWN')
]

choices = [
    'DECLINED',
    'NR',
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['PartnerHIVStatus'] = np.select(conditions, choices, default='None')
hts['PartnerHIVStatus'] = hts['PartnerHIVStatus'].replace({'None': np.nan})


In [114]:
display(hts['PartnerHIVStatus'].value_counts(dropna=False))

,count
PartnerHIVStatus,
UNKNOWN,1035971
NEGATIVE,597770
NaN,413729
POSITIVE,50679
DECLINED,14901
NR,6018


In [115]:
#Check for positivity rate
tab = pd.crosstab(
    hts['PartnerHIVStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 14% positivity for those  partnersHIV status was positive and 3% for those who did not know the partners status

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
PartnerHIVStatus,,,
DECLINED,14382,519,3.482988
NEGATIVE,593259,4511,0.754638
NR,5963,55,0.913925
POSITIVE,43721,6958,13.729553
UNKNOWN,1002971,33000,3.185417
NaN,405251,8478,2.049167


In [116]:
#Quick check on the shape of dataset
rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2119068 rows and 56 columns in the hts table


UnknownStatusPartner

In [117]:
display(hts['UnknownStatusPartner'].value_counts(dropna=False))

,count
UnknownStatusPartner,
NO,1133785
YES,489959
NaN,436553
Yes,29465
Declined to answer,19637
No,9669


In [118]:
# Unprotected Sex with partner with unknown HIV status
hts['UnknownStatusPartner'] = hts['UnknownStatusPartner'].str.upper()

conditions = [
    hts['UnknownStatusPartner'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <15) & hts['UnknownStatusPartner'].notna(),
    hts['UnknownStatusPartner'].eq('YES'),
    hts['UnknownStatusPartner'].eq('NO')
]

choices = [
    'DECLINED',
    'NR',
    'YES',
    'NO'
]
hts['UnknownStatusPartner'] = np.select(conditions, choices, default='None')
hts['UnknownStatusPartner'] = hts['UnknownStatusPartner'].replace({'None': np.nan})
#display(hts['UnknownStatusPartner'].value_counts(dropna=False))


In [119]:
display(hts['UnknownStatusPartner'].value_counts(dropna=False))

,count
UnknownStatusPartner,
NO,1139452
YES,518494
NaN,436553
DECLINED,19637
NR,4932


In [120]:
#Check positivity rate
tab = pd.crosstab(
    hts['UnknownStatusPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # ~4% positivity rate for those who do not know their partners status

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
UnknownStatusPartner,,,
DECLINED,18996,641,3.264246
NO,1116461,22991,2.017724
NR,4883,49,0.993512
YES,498118,20376,3.929843
NaN,427089,9464,2.167893


In [121]:
# Calculate the number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO'
num_USP_no_sex= hts[
    (hts['UnknownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': {num_USP_no_sex}")

Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': 0


KnownStatusPartner

In [122]:
display(hts['KnownStatusPartner'].value_counts(dropna=False))

,count
KnownStatusPartner,
NO,1490452
NaN,551236
Declined to answer,38192
No,34265
Yes,4869
YES,54


In [123]:
# Unprotected Sex with partner with known HIV status
hts['KnownStatusPartner'] = hts['KnownStatusPartner'].astype(str).str.upper

condtions = [
    hts['KnownStatusPartner'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <15) & hts['KnownStatusPartner'].notna(),
    hts['KnownStatusPartner'].eq('YES'),
    hts['KnownStatusPartner'].eq('NO')
]

choices = [
    'DECLINED',
    'NR',
    'YES',
    'NO'
]

hts['KnownStatusPartner'] = np.select(conditions, choices, default='None')
hts['KnownStatusPartner'] = hts['KnownStatusPartner'].replace({'None': np.nan})
#display(hts['KnownStatusPartner'].value_counts(dropna=False))

In [124]:
display(hts['KnownStatusPartner'].value_counts(dropna=False))

,count
KnownStatusPartner,
NO,1139452
YES,518494
NaN,436553
DECLINED,19637
NR,4932


In [125]:
#check positivity rate
tab = pd.crosstab(
    hts['KnownStatusPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# ~ higher positivity amongst those who know their partners HIV status

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
KnownStatusPartner,,,
DECLINED,18996,641,3.264246
NO,1116461,22991,2.017724
NR,4883,49,0.993512
YES,498118,20376,3.929843
NaN,427089,9464,2.167893


In [126]:
# Calculate the number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO'
num_UP_no_sex= hts[
    (hts['KnownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': {num_UP_no_sex}")

Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': 0


# Key Population  Features

Population Type

In [127]:
## Population Type -
# If Null, then NA
hts['PopulationType'] = hts['PopulationType'].fillna('NA')

display(hts['PopulationType'].value_counts(dropna=False))

,count
PopulationType,
General Population,1886051
Key Population,153974
Priority Population,70037
NA,8920
Vulnerable Population,86


In [128]:
#Rename Vulnerable_Pop to Key_Pop because it has very few records
hts['PopulationType']= hts['PopulationType'].replace(
    'Vulnerable Population',
    'Key Population'
)
hts['PopulationType'].value_counts(dropna=False)

,count
PopulationType,
General Population,1886051
Key Population,154060
Priority Population,70037
NA,8920


In [129]:
# Label as GP, KP, Priority

# Convert PopulationType to uppercase
hts['PopulationType'] = hts['PopulationType'].str.upper()

# Map PopulationType values to 'GP', 'KP', or 'PRIORITY'
conditions = [
    hts['PopulationType'].eq('GENERAL POPULATION'),
    hts['PopulationType'].eq('KEY POPULATION'),
    hts['PopulationType'].eq('PRIORITY POPULATION')
]
choices = ['GP', 'KP', 'PRIORITY']
hts['PopulationType'] = np.select(conditions, choices, default=hts['PopulationType'])


In [130]:
display(hts['PopulationType'].value_counts(dropna=False))

,count
PopulationType,
GP,1886051
KP,154060
PRIORITY,70037
NA,8920


Key Population

In [131]:
display(hts['KeyPopulation'].value_counts(dropna=False))

,count
KeyPopulation,
NaN,1969869
Female sex worker,85058
Men who have sex with men,36239
People in prison and other closed settings,23603
People who inject drugs,4255
Other,44


In [132]:

#KPs under age of 15
hts.loc[
    (hts['PopulationType'] == 'KP') &
    (hts['Age'] < 15),
    ['Age', 'PopulationType', 'KeyPopulation']
].head()

,Age,PopulationType,KeyPopulation
15964,14.0,KP,People in prison and other closed settings
16155,9.0,KP,NaN
16634,1.0,KP,NaN
17499,7.0,KP,NaN
20741,2.0,KP,NaN


In [133]:
# We're going to use NR for not relevant throughout
# Clean labels for KPs. Create other for rare values

hts['KeyPopulation'] = hts['KeyPopulation'].str.upper()

conditions = [
    hts['PopulationType'] != 'KP', # If PopulationType is not KP, set to NR
    (hts['Age']< 15) & hts['KeyPopulation'].notna(),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].eq('FEMALE SEX WORKER'),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('MEN WHO HAVE SEX WITH MEN', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('PEOPLE IN PRISON AND OTHER CLOSED SETTINGS', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('PEOPLE WHO INJECT DRUGS|OTHER', na=False),

]

choices = [
    'NR',
    'NR',
    'FSW',
    'MSM',
    'PRISONER',
    'PWID'
]

hts['KeyPopulation'] = np.select(conditions, choices, default= 'None') #Combined Other with PWID because Other had 45records
hts['KeyPopulation'] = hts['KeyPopulation'].replace({'None': np.nan})

#display(hts['KeyPopulation'].value_counts(dropna=False))


In [134]:
display(hts['KeyPopulation'].value_counts(dropna=False))

,count
KeyPopulation,
NR,1965035
FSW,85052
MSM,36225
PRISONER,23587
NaN,4870
PWID,4299


In [135]:
#Check for positivity rate

tab = pd.crosstab(
    hts['KeyPopulation'],
    hts['FinalTestResult'],
    dropna=False
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #Higher positivity rate in MSM


FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
KeyPopulation,,,
FSW,84261,791,0.930019
MSM,35737,488,1.347136
NR,1913216,51819,2.637052
PRISONER,23280,307,1.301564
PWID,4277,22,0.511747
NaN,4776,94,1.930185


Priority Population

In [136]:
display(hts['PriorityPopulation'].value_counts(dropna=False))

,count
PriorityPopulation,
NaN,2055324
Adolescent and young girls,22774
Fisher folk,18701
Prisoner,18540
Truck driver,2117
Young women aged 15-24 years,1209
Military and other uniformed services,365
Families and children living on the streets,13
People who abuse alcohol and other drugs,12


In [144]:
## Priority Population
#display(hts['PriorityPopulation'].value_counts(dropna=False))

hts['PriorityPopulation'] = hts['PriorityPopulation'].str.upper()

conditions = [
    hts['PopulationType'] != 'PRIORITY', # If PopulationType is not PRIORITY, set to NR
    (hts['Age']< 15) & hts['PriorityPopulation'].notna(),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].eq('FISHER FOLK'),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].eq('ADOLESCENT AND YOUNG GIRLS|YOUNG WOMEN AGED 15-24 YEARS'),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].eq('PRISONER'),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].eq('TRUCK DRIVER'),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].eq('MILITARY AND OTHER UNIFORMED SERVICES|FAMILIES AND CHILDREN LIVING ON THE STREETS|PEOPLE WHO ABUSE ALCOHOL AND OTHER DRUGS|SERVICE MEN AND WOMEN AND THEIR FAMILIES|WIDOWS AND WIDOWERES|OTHERS|ORPHANS AND VULNERABLE CHILDREN')
]

choices = [
    'NR',
    'NR',
    'FISHERMEN',
    'AGYW',
    'PRISONERS',
    'TRUCKDRIVERS',
    'OTHER'
]

hts['PriorityPopulation'] = np.select(conditions, choices, default='None')
hts['PriorityPopulation'] = hts['PriorityPopulation'].replace({'None': np.nan})

# Display the new distribution of PriorityPopulation
display(hts['PriorityPopulation'].value_counts(dropna=False))

,count
PriorityPopulation,
NR,2049435
NaN,69633


In [ ]:
#Check for positivity rate
pd.crosstab(
    hts['PriorityPopulation'],
    hts['FinalTestResult'],
    dropna=False
)

FinalTestResult,NEGATIVE,POSITIVE,NaN
PriorityPopulation,,,
AGYW,23789,163,223
FISHERMEN,18275,417,471
NAN,6333,38,51
NR,1997441,53239,57433
OTHER,390,13,6
PRISONER,18268,268,162
TRUCK,2064,52,13


In [ ]:
# Fix population type
hts_population_conditions = [
    (hts['KeyPopulation'] == 'NR') & (hts['PriorityPopulation'] == 'NR'),
    (hts['KeyPopulation'] != 'NR'),
    (hts['PriorityPopulation'] !='NR')

]

hts_population_choices = [
    'GP',
    'KP',
    'PRIORITY'
]

hts['PopulationType'] = np.select(
    hts_population_conditions,
    hts_population_choices,
    default=hts['PopulationType']
)

# Display the new distribution of PopulationType
display(hts['PopulationType'].value_counts(dropna=False)) # If  KP and Priority are NR, then they are GP, if they are Key Pop, classify as Kp same with Priority pop

,count
PopulationType,
GP,1950918
KP,157195
PRIORITY,70996


IsHealthCareWorker

In [145]:
display(hts['IsHealthWorker'].value_counts(dropna=False))

,count
IsHealthWorker,
No,1899571
NaN,134324
Yes,85173


In [146]:
## Is Health Worker - what should 0 be? No or Null?
#display(hts['IsHealthWorker'].value_counts(dropna=False))

# If <17, set to not relevant
hts['IsHealthWorker'] = hts['IsHealthWorker'].str.upper()

conditions = [
    (hts['Age'] <17) & hts['IsHealthWorker'].notna(),
    hts['IsHealthWorker'].eq('YES'),
    hts['IsHealthWorker'].eq('NO')
]
choices = [
    'NR',
    'YES',
    'NO'
]

hts['IsHealthWorker'] = np.select(conditions, choices, default='None')
hts['IsHealthWorker'] = hts['IsHealthWorker'].replace({'None': np.nan})


In [147]:
display(hts['IsHealthWorker'].value_counts(dropna=False))

,count
IsHealthWorker,
NO,1882225
NaN,134324
YES,84718
NR,17801


In [148]:

#Check for positivity rate
tab = pd.crosstab(
    hts['IsHealthWorker'],
    hts['FinalTestResult']
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
) * 100 #There is a very low positivity rates amongst healthcare workers~ 2.24%

# TB Risk Features

In [149]:
hts['TBStatus'].value_counts(dropna=False)
hts['TBStatus'] = hts['TBStatus'].str.upper()
hts['TBStatus'].value_counts(dropna=False)

,count
TBStatus,
NO TB SIGNS,1650972
NaN,322526
PRESUMED TB,140857
TB CONFIRMED,4706
TB SCREENING NOT DONE,7


TB Status

In [150]:

#Convert TBStatus to uppercase then map values
hts['TBStatus'] = hts['TBStatus'].str.upper()

conditions = [
    hts['TBStatus'].eq('NO TB SIGNS'),
    hts['TBStatus'].eq('PRESUMED TB'),
    hts['TBStatus'].eq('TB CONFIRMED'),
    hts['TBStatus'].eq('TB SCREENING NOT DONE	')
]

choices = [
    'NOTBSIGNS',
    'PRESUMED',
    'CONFIRMED',
    'NOTDONE'
]

hts['TBStatus'] = np.select(conditions, choices, default='None')
hts['TBStatus'] = hts['TBStatus'].replace({'None': np.nan})

In [151]:
hts['TBStatus'].value_counts(dropna=False)

,count
TBStatus,
NOTBSIGNS,1650972
NaN,322533
PRESUMED,140857
CONFIRMED,4706


In [152]:
#Check positivity rate
tab = pd.crosstab(
    hts['TBStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100

display(tab) # 12% positivity rate in those with confirmed TB Status

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
TBStatus,,,
CONFIRMED,4154,552,11.729707
NOTBSIGNS,1618552,32420,1.963692
PRESUMED,128234,12623,8.961571
NaN,314607,7926,2.457423


Screened TB

In [ ]:
hts['ScreenedTB'].value_counts(dropna=False)

,count
ScreenedTB,
YES,1807847
NO,196912
Yes,57996
NaN,37319
No,14931
Declined to answer,4063


In [ ]:
#Convert TBStatus to uppercase then map values
hts['ScreenedTB'] = hts['ScreenedTB'].str.upper()

conditions = [
    hts['ScreenedTB'].eq('YES'),
    hts['ScreenedTB'].eq('NO')
]

choices = [
    'YES',
    'NO'
]

hts['ScreenedTB'] = np.select(conditions, choices, default='None')
hts['ScreenedTB'] = hts['ScreenedTB'].replace({'None': np.nan})


In [ ]:
hts['ScreenedTB'].value_counts(dropna=False)

,count
ScreenedTB,
YES,1865843
NO,211843
NaN,41382


In [ ]:
#Check positivity rate
tab = pd.crosstab(
    hts['ScreenedTB'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100

display(tab) #High positivity rate in those who screened for TB

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ScreenedTB,,,
NO,207806,4037,1.905657
YES,1818533,47310,2.535583
NaN,39208,2174,5.253492


# Violence and vulnerable Features

ExperiencedViolenceScreening

In [ ]:
display(hts['ExperiencedViolenceScreening'].value_counts(dropna=False))

,count
ExperiencedViolenceScreening,
NO,1803672
YES,185140
NaN,92329
No,34294
Yes,3565
Declined to answer,68


In [ ]:
# Recently Experienced GBV
hts['ExperiencedViolenceScreening'] = hts['ExperiencedViolenceScreening'].astype(str).str.upper()


conditions_gbv = [
    hts['ExperiencedViolenceScreening'].eq('DECLINED TO ANSWER'),
    (hts['Age'] <9) & hts['ExperiencedViolenceScreening'].notna(), # Not relevant for children
    hts['ExperiencedViolenceScreening'].eq('YES'),
    hts['ExperiencedViolenceScreening'].eq('NO')
]

choices_gbv = [
    'DECLINED',
    'NR',
    'YES',
    'NO'
]

hts['ExperiencedViolenceScreening'] = np.select(conditions_gbv, choices_gbv, default='None')
hts['ExperiencedViolenceScreening'] = hts['ExperiencedViolenceScreening'].replace({'None': np.nan})

# Create cross-tabulation tables (proportions)

prop_table_gbv_sex = pd.crosstab(hts['Sex'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_age_gbv = pd.crosstab(hts['Age'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_gbv_poptype = pd.crosstab(hts['PopulationType'], hts['ExperiencedViolenceScreening'], normalize='index')

#display(hts['ExperiencedViolenceScreening'].value_counts(dropna=False))



In [ ]:
#Check for positivity
tab = pd.crosstab(
    hts['ExperiencedViolenceScreening'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# ~3% positivity rate in those who experienced violence

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ExperiencedViolenceScreening,,,
DECLINED,68,0,0.000000
NO,1792736,44363,2.414840
NR,47738,929,1.908891
YES,182684,5887,3.121901
NaN,42321,2342,5.243714


ViolenceScreeningType

In [ ]:
# Three values, create binaries for each

hts['ViolenceScreeningType'] = hts['ViolenceScreeningType'].str.upper()


conditions_sexual = [
    hts['ViolenceScreeningType'].str.contains('SEXUAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_sexual = ['YES', 'NO', 'NAN']
hts['GBVSexual'] = np.select(conditions_sexual, choices_sexual, default='NR')


conditions_physical = [
    hts['ViolenceScreeningType'].str.contains('PHYSICAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_physical = ['YES', 'NO', 'NAN']
hts['GBVPhysical'] = np.select(conditions_physical, choices_physical, default='NR')


conditions_emotional = [
    hts['ViolenceScreeningType'].str.contains('EMOTIONAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_emotional = ['YES', 'NO', 'NAN']
hts['GBVEmotional'] = np.select(conditions_emotional, choices_emotional, default='NR')


In [ ]:
crosstab_Emotional_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVEmotional'], dropna=False) # 8127 experienced VS but not GBVEmotional
crosstab_Physical_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVPhysical'], dropna=False) #16958 experienced VS but not GBVPhysical
crosstab_Sexual_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVSexual'], dropna=False) #2426 experienced VS but not GBVSexual

#Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': 0
#Number of individuals who experienced GBV but did not report any specific GBV type: 37504

In [ ]:
# Count rows where ExperiencedViolenceScreening is NA or 'NO', and no specific GBV type is 'YES'
num_rows_no_gbv_flags = hts[
    (hts['ExperiencedViolenceScreening'].isna() | (hts['ExperiencedViolenceScreening'] == 'NO')) &
    (hts['GBVEmotional'] == 'YES') &
    (hts['GBVPhysical'] == 'YES') &
    (hts['GBVSexual'] == 'YES')
].shape[0]

print(f"Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': {num_rows_no_gbv_flags}")

Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': 0


In [ ]:
#Number of individuals who experienced GBV but did not report any specific GBV type
num_inconsistent_gbv_screening = hts[
    (hts['ExperiencedViolenceScreening'] == 'YES') &
    (hts['GBVEmotional'] != 'YES') &
    (hts['GBVPhysical'] != 'YES') &
    (hts['GBVSexual'] != 'YES')
].shape[0]

print(f"Number of individuals who experienced GBV but did not report any specific GBV type: {num_inconsistent_gbv_screening}")

Number of individuals who experienced GBV but did not report any specific GBV type: 136712


In [ ]:
#positivity rates for violence screening
tab = pd.crosstab(
    hts['ExperiencedViolenceScreening'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3% positivity rate on those who experienced violence screening

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ExperiencedViolenceScreening,,,
DECLINED,68,0,0.000000
NO,1792736,44363,2.414840
NR,47738,929,1.908891
YES,182684,5887,3.121901
NaN,42321,2342,5.243714


In [ ]:
#positivity rates for violence screening
tab = pd.crosstab(
    hts['GBVEmotional'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
GBVEmotional,,,
NAN,245694,8209,3.233124
NO,1792735,44363,2.414841
YES,27118,949,3.381195


In [ ]:
#positivity rates for violence screening
tab = pd.crosstab(
    hts['GBVSexual'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
GBVSexual,,,
NAN,230502,7463,3.136175
NO,1792736,44363,2.414840
YES,42309,1695,3.851923


In [ ]:
#positivity rates for violence screening
tab = pd.crosstab(
    hts['GBVPhysical'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
GBVPhysical,,,
NAN,272772,9157,3.247981
NO,1792736,44363,2.414840
YES,39,1,2.500000


PatientDisabled

In [ ]:
display(hts['PatientDisabled'].value_counts(dropna=False))

,count
PatientDisabled,
Yes,1859456
No,259612


In [ ]:
# PatientDisabled
hts['PatientDisabled'] = hts['PatientDisabled'].str.upper()

cross_tab_patient_disabled_population_type = pd.crosstab(hts['PatientDisabled'], hts['PopulationType']) #64376 are GP, 43141 are KP and 9817 are Priority

In [ ]:
#Check positivity rate
tab = pd.crosstab(
    hts['PatientDisabled'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #Positivity rate is lower in disabled patients

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
PatientDisabled,,,
NO,257931,1681,0.647505
YES,1807616,51840,2.787912


# PrEP Features

CurrentlyOnPrep

In [ ]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'MonthsSinceLastTest',
       'FinalTestResult', 'ClientTestedAs', 'CoupleDiscordant',
       'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County',
       'SubCounty', 'Latitude', 'Longitude', 'SDP', 'EMR', 'KEPH_Level',
       'Ward', 'VDate', 'DOB', 'Age', 'cohort', 'GBVSexual', 'GBVPhysical',
       'GBVEmo

In [ ]:
display(hts['CurrentlyOnPrep'].value_counts(dropna=False))

,count
CurrentlyOnPrep,
NO,1074072
NaN,982063
YES,50294
Declined to answer,7157
No,4568
Yes,914


In [ ]:
#Currently on PrEP
hts['CurrentlyOnPrep'] = hts['CurrentlyOnPrep'].str.upper()
conditions = [
    hts['CurrentlyOnPrep'].eq(  'DECLINED TO ANSWER'),
    hts['Age'] <15 & hts['CurrentlyOnPrep'].notna() ,
    hts['CurrentlyOnPrep'].eq('YES'),
    hts['CurrentlyOnPrep'].eq('NO')
]
choices = [
    'DECLINED',
    'NR',
    'YES',
    'NO'
]
hts['CurrentlyOnPrep'] = np.select(conditions, choices, default='None')
hts['CurrentlyOnPrep'] = hts['CurrentlyOnPrep'].replace({'None': np.nan})

prep_by_sex = hts.groupby('Sex')['CurrentlyOnPrep'].value_counts(normalize=True).unstack(fill_value=0)


In [ ]:
display(hts['CurrentlyOnPrep'].value_counts(dropna=False))

,count
CurrentlyOnPrep,
NO,1078640
NaN,982063
YES,51208
DECLINED,7157


In [ ]:
#check positivity
tab = pd.crosstab(
    hts['CurrentlyOnPrep'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2% positivity rate amongst those not currently on PrEP

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
CurrentlyOnPrep,,,
DECLINED,6990,167,2.333380
NO,1052301,26339,2.441871
YES,50861,347,0.677628
NaN,955395,26668,2.715508


ReceivedServices

In [ ]:
# Received Services - prep, pep, tb, sti
hts['ReceivedServices'].value_counts(dropna=False)

,count
ReceivedServices,
"PEP,PrEP,STI",1052829
NaN,901635
PrEP,79888
TB,22863
"PEP,STI",20197
STI,15749
"PEP,PrEP",11405
PEP,7586
"PrEP,STI",6915


In [ ]:
# Four values, create binaries for each
hts['ReceivedServices'] = hts['ReceivedServices'].str.upper()

def service_flag(keyword):
    return np.select(
        [
            hts['ReceivedServices'].str.contains(
                rf'(?:^|,){keyword}(?:,|$)',
                na=False
            ),
            hts['ReceivedServices'].notna()
        ],
        [
            'YES',
            'NO'
        ],
        default=None
    )
for service in ['PREP', 'PEP', 'TB', 'STI']:
    hts[f'Received{service}'] = service_flag(service)

In [ ]:
for service in ['PREP', 'PEP', 'TB', 'STI']:
    print(f"\n{service}")
    display(hts[f'Received{service}'].value_counts(dropna=False))


PREP


,count
ReceivedPREP,
YES,1151038
None,901635
NO,66395



PEP


,count
ReceivedPEP,
YES,1092018
None,901635
NO,125415



TB


,count
ReceivedTB,
NO,1194569
None,901635
YES,22864



STI


,count
ReceivedSTI,
YES,1095691
None,901635
NO,121742


In [ ]:
#Check positivity rate PrEP
tab = pd.crosstab(
    hts['ReceivedPrEP'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.3% positivity rate for those who had received PrEP

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ReceivedPrEP,,,
False,941485,26545,2.742167
True,1124062,26976,2.343624


In [ ]:
#Check positivity rate PEP
tab = pd.crosstab(
    hts['ReceivedPEP'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.4% positivity rate for those who had received PEP

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ReceivedPEP,,,
NO,122574,2841,2.265279
YES,1065877,26141,2.393825
NaN,877096,24539,2.721611


In [ ]:
#check positivity rate for TB
tab = pd.crosstab(
    hts['ReceivedTB'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 4% of positivity rate for those who received TB services

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ReceivedTB,,,
NO,1166520,28049,2.348044
YES,21931,933,4.080651
NaN,877096,24539,2.721611


In [ ]:
#check positivity rate for STI
tab = pd.crosstab(
    hts['ReceivedSTI'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.4% positivity rate for those who received STI services


FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
ReceivedSTI,,,
NO,119352,2390,1.963168
YES,1069099,26592,2.426962
NaN,877096,24539,2.721611


In [ ]:
#Quick Check
# no one currently on PrEP said they haven't received prep services
crosstab_prep_status = pd.crosstab(hts['ReceivedPrEP'], hts['CurrentlyOnPrep'], dropna=False)

# 24214 patients with presumed TB said they haven't received TB services
crosstab_TB_status = pd.crosstab(hts['ReceivedTB'], hts['TBStatus'], dropna=False)

# Sexual Exposure Risk Features

EverHadSex

In [ ]:
display(hts['EverHadSex'].value_counts(dropna=False))

,count
EverHadSex,
YES,1822753
NaN,293451
NO,2864


In [ ]:
## For all sexual practice variables, if under 9 years old, then classify as NR

hts['EverHadSex'] = hts['EverHadSex'].astype(str).str.upper()

conditions = [
    hts['Age'] <15,
    hts['EverHadSex'].str.contains('YES', na=False),
    hts['EverHadSex'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['EverHadSex'] = np.select(conditions, choices, default='None')
hts['EverHadSex'] = hts['EverHadSex'].replace({'None': np.nan})

#display(hts['EverHadSex'].value_counts(dropna=False))

In [ ]:
display(hts['EverHadSex'].value_counts(dropna=False))

,count
EverHadSex,
YES,1819357
NaN,207620
NR,89228
NO,2863


In [ ]:
#Check for positivity rate
tab= pd.crosstab(
    hts['EverHadSex'],
    hts['FinalTestResult'],
    dropna=False
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #positivity rate of 2.6% for those who have ever had sex

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
EverHadSex,,,
NO,2822,41,1.432064
NR,87672,1556,1.743847
YES,1772346,47011,2.583935
NaN,202707,4913,2.366342


Sexually Active

In [ ]:
display(hts['SexuallyActive'].value_counts(dropna=False))

,count
SexuallyActive,
YES,1587380
NAN,315605
NO,212434
DECLINED TO ANSWER,3649


In [ ]:
# Sexually Active
#display(hts['SexuallyActive'].value_counts(dropna=False))


hts['SexuallyActive'] = hts['SexuallyActive'].astype(str).str.upper()
conditions = [
    hts['SexuallyActive'].eq('DECLINED TO ANSWER'),
    (hts['Age']<15) & hts['SexuallyActive'].notna(),
    hts['SexuallyActive'].eq('YES'),
    hts['SexuallyActive'].eq('NO')
  ]


choices = [
            'DECLINED',
            'NR',
            'YES',
            'NO'
 ]

hts['SexuallyActive']= np.select(conditions, choices, default='None')
hts['SexuallyActive'] = hts['SexuallyActive'].replace({'None': np.nan})
#display(hts['SexuallyActive'].value_counts(dropna=False))

In [ ]:
display(hts['SexuallyActive'].value_counts(dropna=False))

,count
SexuallyActive,
YES,1578195
NaN,287751
NO,160720
NR,88753
DECLINED,3649


In [ ]:
#Check for positivity rate
#make the options uppercase
hts['SexuallyActive']=hts['SexuallyActive'].astype(str).str.upper()

tab = pd.crosstab(
    hts['SexuallyActive'],
    hts['FinalTestResult'],
    dropna=False
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # ~2.7% positivity rate for those who have ever had sex and ~4% for those who declined to answer

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
SexuallyActive,,,
DECLINED,3505,144,3.946287
NAN,281183,6568,2.282529
NO,157444,3276,2.038328
NR,87216,1537,1.731772
YES,1536199,41996,2.661015


In [ ]:
display(hts.columns)

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'MonthsSinceLastTest',
       'FinalTestResult', 'ClientTestedAs', 'CoupleDiscordant',
       'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County',
       'SubCounty', 'Latitude', 'Longitude', 'SDP', 'EMR', 'KEPH_Level',
       'Ward', 'VDate', 'DOB', 'Age', 'cohort', 'GBVSexual', 'GBVPhysical',
       'GBVEmo

# Traditional Exposure Risk Features

TraditionalProcedures

In [ ]:
display(hts['TraditionalProcedures'].value_counts(dropna=False))

,count
TraditionalProcedures,
No,1911648
NaN,108559
Yes,98861


In [ ]:
#Traditional Procedures
hts['TraditionalProcedures'] = hts['TraditionalProcedures'].str.upper()

In [ ]:
#check positivity rate
tab = pd.crosstab(
    hts['TraditionalProcedures'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3% positivity for those who had traditional procedures

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
TraditionalProcedures,,,
NO,1865436,46212,2.417391
YES,95795,3066,3.101324
NaN,104316,4243,3.908474


# Family Risk Features

MothersStatus

In [ ]:
(hts['MothersStatus'].value_counts(dropna=False))

,count
MothersStatus,
NaN,2064631
Positive,28459
Negative,16717
Unknown,9261


In [ ]:
#Mother's HIV Status - asked of kids only
hts['MothersStatus'] = hts['MothersStatus'].str.upper()

conditions = [
    (hts['Age'] >= 15) & hts['MothersStatus'].isna(),
    hts['MothersStatus'].str.contains('POSITIVE', na=False),
    hts['MothersStatus'].str.contains('NEGATIVE', na=False),
    hts['MothersStatus'].str.contains('UNKNOWN', na=False)
]

choices = [
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['MothersStatus'] = np.select(conditions, choices, default='None')
hts['MothersStatus'] = hts['MothersStatus'].replace({'None': np.nan})
# display(hts['MothersStatus'].value_counts(dropna=False))

In [ ]:
display(hts['MothersStatus'].value_counts(dropna=False))

,count
MothersStatus,
NR,2029389
NaN,35242
POSITIVE,28459
NEGATIVE,16717
UNKNOWN,9261


In [ ]:
#check positivity rates
tab = pd.crosstab(
    hts['MothersStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ higher positivity rate amongst those whose mothers status was positive

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
MothersStatus,,,
NEGATIVE,16661,56,0.334988
NR,1977452,51937,2.559243
POSITIVE,27612,847,2.976211
UNKNOWN,9096,165,1.781665
NaN,34726,516,1.464162


# Distribution by day of week

In [ ]:
#Let day of week (skip month because we don't have a whole year or multiple years)
hts['dayofweek'] = hts['VDate'].dt.day_name().str.upper() # Monday=0, Sunday=6

display(hts['dayofweek'].value_counts(dropna=False))

,count
dayofweek,
TUESDAY,436590
MONDAY,426971
WEDNESDAY,410551
THURSDAY,404683
FRIDAY,337948
SATURDAY,62874
SUNDAY,39451


In [ ]:
#Check positivity by day of week
tab = pd.crosstab(
    hts['dayofweek'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #positivity rate higher on Monday where majority of teste are done

FinalTestResult,NEGATIVE,POSITIVE,PositivityRate
dayofweek,,,
FRIDAY,330265,7683,2.273427
MONDAY,414024,12947,3.032290
SATURDAY,62234,640,1.017909
SUNDAY,39146,305,0.773111
THURSDAY,394848,9835,2.430297
TUESDAY,424853,11737,2.688335
WEDNESDAY,400177,10374,2.526848


In [ ]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'LoadDate_x',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'PriorityPopulationType',
       'MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR', 'EMR_Status', 'HTS_

In [ ]:
cols_to_drop = ["Dob", "DOB", "present", "cohort", "VDate",
                "CurrentlyOnPep","DateTestedProvider",  'Disability', 'DisabilityType','LoadDate_x','KMPDC_reg_no','ReceivedPEP','EMR_Status', 'HTS_Use',
                'HTS_Deployment', 'Project', 'LoadDate_y', 'InfrastructureType',
                'KEPH_Level', 'KMPDC_reg_no', 'Ward',]

hts = hts.drop(columns=cols_to_drop, errors='ignore')

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after dropping columns") #There are 2179109 rows and 58 columns in the hts table after dropping columns

There are 2179109 rows and 58 columns in the hts table after dropping columns


In [ ]:
#download the hts file
hts.to_csv('hts_cleaned.csv', index=False)

from google.colab import files
files.download('hts_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
display(hts.columns)

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Sex', 'MaritalStatus', 'PatientDisabled',
       'EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County',
       'SubCounty', 'Owner', 'Latitude', 'Longitude', 'SDP', 'SDP_Agency',
       'EMR', 'CoupleDiscordant', 'Age', 'ReceivedPrEP', 'ReceivedTB',
       'Receive

In [ ]:
#Check missingness in hts dataset
missing_values = hts.isnull().sum()
missing_values_percentage = (missing_values / len(hts)) * 100
missing_values_df = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_values_percentage})
missing_values_df

,Missing Values,Percentage
SiteCode,0,0.000000
PatientPk,0,0.000000
VisitDate,0,0.000000
PopulationType,0,0.000000
KeyPopulation,0,0.000000
PriorityPopulation,0,0.000000
IsHealthWorker,0,0.000000
RelationshipWithContact,1962670,90.067546
TestedHIVBefore,9440,0.433205
ResultOfHIV,930752,42.712503


In [ ]:
#Check for variance
cols = [
       'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Sex', 'MaritalStatus', 'PatientDisabled',
       'EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'PriorityPopulationType',  'CoupleDiscordant', 'Age', 'ReceivedPrEP', 'ReceivedTB',
       'ReceivedSTI', 'GBVSexual', 'GBVPhysical', 'GBVEmotional', 'dayofweek'
]

for col in cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(hts[col].value_counts(dropna=False, normalize=True) * 100)


Column: PopulationType
PopulationType
GP          89.528243
KP           7.213728
PRIORITY     3.258029
Name: proportion, dtype: float64

Column: KeyPopulation
KeyPopulation
NR          92.786272
FSW          4.009024
MSM          1.679999
PRISONER     1.095998
NAN          0.229360
PWID         0.199348
Name: proportion, dtype: float64

Column: PriorityPopulation
PriorityPopulation
NR           96.741971
AGYW          1.109398
FISHERMEN     0.879396
PRISONER      0.858057
NAN           0.294708
TRUCK         0.097700
OTHER         0.018769
Name: proportion, dtype: float64

Column: IsHealthWorker
IsHealthWorker
NO     89.506307
NR      6.497610
YES     3.996083
Name: proportion, dtype: float64

Column: RelationshipWithContact
RelationshipWithContact
NaN                                             90.067546
Sexual Contact                                   5.522486
Social Contact                                   3.709957
Sexual Contact,Social Contact                    0.359046
Needle 

In [ ]:
cols_to_drop = ['MFL_Code']

hts = hts.drop(columns=cols_to_drop, errors='ignore')

# Data Imputation

                    Let's join with GIS variables

In [ ]:
!pip install pyreadr
import pyreadr

In [ ]:
# Read the rds dataset
result = pyreadr.read_r('gis_features_iit.rds')
gis = result[None]

display(gis.head())

In [ ]:
if 'Latitude' in gis.columns:
    gis = gis.drop(columns=['Latitude'])
if 'Longitude' in gis.columns:
    gis = gis.drop(columns=['Longitude'])

    gis.drop_duplicates(subset=['FacilityCode'], keep='first', inplace=True)

    # Select only numeric columns for mean imputation
    numeric_cols = gis.select_dtypes(include=np.number).columns
    gis[numeric_cols] = gis[numeric_cols].fillna(gis[numeric_cols].mean())

    hts['SiteCode'] = hts['SiteCode'].astype(str)

    hts = pd.merge(hts, gis, left_on="SiteCode", right_on="FacilityCode", how="inner").drop(columns=['SiteCode_y'])

    hts.replace("", np.nan, inplace=True)

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after joining with GIS variables")

                                   Missing data imputation

In [ ]:
# We'll create two versions of the dataset
# 1) Keep missing values missing. Some ML models have sophisticated in-built ways of
# dealing with missing values, particularly XGBoost. For others, we'll need to impute.
# 2) Simple imputation - mean and mode. However, we'll only do this for variables that
# are present at least half the time. For very sparse variables, we don't have enough
# of a basis to impute. For these, we'll give missing values a label of MISSING


In [ ]:
# For all imputation, we're going to learn how to impute from the train set only to
# avoid any leakage. So, first step is to split dataset into train-eval-test.
# do 60-20-20 train-val-test split


In [ ]:
import random

random.seed(2231)
np.random.seed(2231)

In [ ]:
from sklearn.model_selection import train_test_split

# Drop rows where 'FinalTestResult' is NaN before splitting for stratification..
hts_cleaned = hts.dropna(subset=['FinalTestResult']).copy()

hts_train, hts_temp_test = train_test_split(hts_cleaned, test_size=0.4, random_state=2231, stratify=hts_cleaned['FinalTestResult'])
hts_val, hts_test = train_test_split(hts_temp_test, test_size=0.5, random_state=2231, stratify=hts_temp_test['FinalTestResult'])

print(f"Original HTS shape: {hts.shape}")
print(f"HTS shape after dropping NaNs in FinalTestResult (for splitting): {hts_cleaned.shape}")
print(f"Training set shape: {hts_train.shape}")
print(f"Validation set shape: {hts_val.shape}")
print(f"Test set shape: {hts_test.shape}")

In [ ]:
# Create sparse versions of the datasets by simply copying the split DataFrames
sparse_train_df = hts_train.copy()
sparse_val_df = hts_val.copy()
sparse_test_df = hts_test.copy()

# Store them in a dictionary for easy access, similar to the R list structure
sparse_datasets = {
    "sparse_train": sparse_train_df,
    "sparse_val": sparse_val_df,
    "sparse_test": sparse_test_df
}

print("Sparse datasets:")
for name, df in sparse_datasets.items():
    print(f"  {name} shape: {df.shape}")

In [ ]:
## Next, simple imputation
# First, identify which variables are present > 50% of the time and should be imputed
# Identify which variables are too sparse and instead will be given a value of MISSING
cols_to_impute = []
cols_to_unknown = []

selected_column_names = list(hts.columns[0:35]) + list(hts.columns[36:48])

for col_name in selected_column_names:
    vals = hts[col_name]

    non_missing_percentage = 100 * (vals.notna().sum() / len(vals))

    #print(col_name)
    #print(round(non_missing_percentage))

    # Condition for cols_to_impute: >50% non-missing AND has some missing values
    if non_missing_percentage > 50 and vals.isna().any():
        cols_to_impute.append(col_name)

    # Condition for cols_to_unknown: >50% missing values
    if (100 - non_missing_percentage) > 50:
        cols_to_unknown.append(col_name)

print(f"\nColumns to impute: {cols_to_impute}")
print(f"Columns to label as unknown: {cols_to_unknown}")

In [ ]:
# This function identifies the mode in one dataframe and imputes to another
# This is critical because we want the mode from the training set only to avoid leakage

def mode_excluding_nr(series):

    filtered_series = series[series != 'NR'].dropna()
    if not filtered_series.empty:
        return filtered_series.mode()[0]
    else:
        return 'UNKNOWN'

def replace_with_mode(df_calc, df_impute, col_name):
    imputation_mode = mode_excluding_nr(df_calc[col_name])

    if pd.api.types.is_numeric_dtype(df_impute[col_name]):
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    else:
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    return df_impute

train_simple_py = sparse_datasets['sparse_train'].copy()
val_simple_py = sparse_datasets['sparse_val'].copy()
test_simple_py = sparse_datasets['sparse_test'].copy()

for col_name in cols_to_impute:
    #print(f"  Imputing column: {col_name}")

    train_simple_py = replace_with_mode(train_simple_py, train_simple_py, col_name)
    val_simple_py = replace_with_mode(train_simple_py, val_simple_py, col_name) # Uses mode from train_simple_py
    test_simple_py = replace_with_mode(train_simple_py, test_simple_py, col_name) # Uses mode from train_simple_py

simple = {
    "simple_train": train_simple_py,
    "simple_val": val_simple_py,
    "simple_test": test_simple_py
}

for name, df_imp in simple.items():
    print(f"Missing values in {name} after translation imputation (first 10):\n{df_imp.isnull().sum().head(10)}")


In [ ]:
# Replace nan with Missing
for col_name in cols_to_unknown:
    # Iterate through all three datasets (train, val, test) and replace NaNs with 'MISSING'
    for df_key in simple:
        simple[df_key][col_name] = simple[df_key][col_name].fillna('MISSING')

#display(simple['simple_train'].head())

In [ ]:
# For simple imputations, add binary variables to indicate whether
# value was missing. This is so that we retain information about missingness.

for col_name in cols_to_impute + cols_to_unknown:
    for df_key, original_df in zip(simple.keys(), [hts_train, hts_val, hts_test]):
        simple[df_key][f'{col_name}_IS_MISSING'] = original_df[col_name].isna().astype(int)


In [ ]:
# For simple, add binary variables to indicate whether
#value was missing. This is so that we retain information about missingness.

sparse_cols_from_train = hts_train.columns[hts_train.isnull().any()].tolist()

# Process simple_train
sparse_binary_train = hts_train[sparse_cols_from_train].notna().astype(int)
sparse_binary_train .columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_train  = sparse_binary_train .reindex(simple['simple_train'].index)
simple['simple_train'] = pd.concat([simple['simple_train'], sparse_binary_train], axis=1)

# Do the same for val set
sparse_binary_val = hts_val[sparse_cols_from_train].notna().astype(int)
sparse_binary_val.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_val = sparse_binary_val.reindex(simple['simple_val'].index)
simple['simple_val'] = pd.concat([simple['simple_val'], sparse_binary_val], axis=1)

# Repeat for test set
sparse_binary_test = hts_test[sparse_cols_from_train].notna().astype(int)
sparse_binary_test.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_test = sparse_binary_test.reindex(simple['simple_test'].index)
simple['simple_test'] = pd.concat([simple['simple_test'], sparse_binary_test], axis=1)



### Saving Processed DataFrames to CSV

In [ ]:
# download sparse datasets to a CSV file
for name, df in sparse_datasets.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All sparse datasets saved as CSVs.")

In [ ]:
# Download Sparse datasets to a CSV file
for name, df in simple.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All simple imputed datasets saved as CSVs.")